# Stage 5 — Frozen three-seed LoRA final validation run

Runs seeds 42, 43, and 44 with the approved frozen LoRA configuration. It reuses the verified batch-32 smoke test, evaluates validation only, caches each adapter and corpus index to Drive, computes 95% percentile-bootstrap confidence intervals, and requires distinct adapter/loss/embedding hashes. The SciFact test split is never loaded.


In [ ]:
%pip uninstall -y -q torchao
%pip install -q "sentence-transformers==6.0.1" "transformers==5.16.1" "faiss-cpu==1.15.0" datasets pandas peft


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import base64
import hashlib
import io
import zipfile
from pathlib import Path

PAYLOAD_SHA256 = 'DD7111FCD92829DA426B538DF0A72E882574FAD30246B6D8CF3AF4B61191D94B'
PAYLOAD_B64 = '...'
payload = base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest().upper() == PAYLOAD_SHA256
SOURCE_DIR = Path('/content/drive/MyDrive/RAFT/stage5_lora/final_source_snapshot')
SOURCE_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    assert all(not Path(name).is_absolute() and '..' not in Path(name).parts for name in archive.namelist())
    archive.extractall(SOURCE_DIR)
print(f'Stage 5 final source ready: {SOURCE_DIR}')
print(f'PAYLOAD_SHA256={PAYLOAD_SHA256}')


In [ ]:
import copy
import json
import os
import subprocess
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/RAFT')
os.environ['HF_HOME'] = '/content/huggingface_cache'
assert torch.cuda.is_available(), 'Select a free Colab GPU runtime before continuing.'
pilot = json.loads((SOURCE_DIR / 'configs/stage5_lora_pilot.json').read_text())
for seed in (42, 43, 44):
    final = json.loads((SOURCE_DIR / f'configs/stage5_lora_final_seed{seed}.json').read_text())
    assert final['training']['random_seed'] == seed
    for section in ('dataset', 'model', 'lora', 'smoke_test', 'efficiency', 'full_finetuning_reference'):
        assert final[section] == pilot[section]
    pilot_training = copy.deepcopy(pilot['training']); pilot_training.pop('random_seed')
    final_training = copy.deepcopy(final['training']); final_training.pop('random_seed')
    assert final_training == pilot_training
required = [
    DRIVE_ROOT / 'stage1_scifact/processed/corpus.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/queries_train.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/qrels_train.tsv',
    DRIVE_ROOT / 'stage1_scifact/processed/queries_validation.jsonl',
    DRIVE_ROOT / 'stage1_scifact/processed/qrels_validation.tsv',
    DRIVE_ROOT / 'stage3_inbatch/final_seed42/results/stage3_final_result.json',
    DRIVE_ROOT / 'stage3_inbatch/final_seed42/best_checkpoint',
    DRIVE_ROOT / 'stage5_lora/smoke_seed42/results/stage5_lora_smoke_result.json',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing restartable Drive inputs: {missing}'
print('FROZEN CONFIG AUDIT: PASSED; ONLY RANDOM SEED VARIES')
print('REUSING VERIFIED STAGE 5 BATCH-32 SMOKE TEST')


In [ ]:
completed = subprocess.run([
    sys.executable,
    str(SOURCE_DIR / 'src/run_stage5_final.py'),
    '--repo-dir', str(SOURCE_DIR),
    '--drive-root', str(DRIVE_ROOT),
], check=True)
print(completed)


In [ ]:
import pandas as pd

FINAL_RESULTS = DRIVE_ROOT / 'stage5_lora/final_results'
per_seed = pd.read_csv(FINAL_RESULTS / 'stage5_final_per_seed_validation.csv')
aggregate = pd.read_csv(FINAL_RESULTS / 'stage5_final_aggregate_validation.csv')
audit_rows = pd.read_csv(FINAL_RESULTS / 'stage5_final_seed_audit.csv')
summary = json.loads((FINAL_RESULTS / 'stage5_final_summary.json').read_text())
audit = json.loads((FINAL_RESULTS / 'stage5_final_seed_audit.json').read_text())
assert summary['test_split_loaded'] is False and audit['test_split_loaded'] is False
print('PER-SEED VALIDATION RESULTS:')
display(per_seed)
print('\nAGGREGATE MEAN AND 95% BOOTSTRAP CI:')
display(aggregate)
print('\nCHECKSUM/DISTINCTNESS AUDIT:')
display(audit_rows)
print(json.dumps(audit['pairwise_embedding_comparisons'], indent=2))
print('ALL LOSS HASHES DISTINCT:', audit['all_loss_hashes_distinct'])
print('ALL EMBEDDING HASHES DISTINCT:', audit['all_embedding_hashes_distinct'])
print('ALL ADAPTER/MODEL HASHES DISTINCT:', audit['all_model_hashes_distinct'])
print('TEST SPLIT LOADED:', summary['test_split_loaded'])


In [ ]:
RESULT_ARCHIVE = FINAL_RESULTS / 'stage5_final_results.zip'
result_names = [
    'stage5_final_per_seed_validation.csv',
    'stage5_final_aggregate_validation.csv',
    'stage5_final_seed_audit.csv',
    'stage5_final_seed_audit.json',
    'stage5_final_summary.json',
]
with zipfile.ZipFile(RESULT_ARCHIVE, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in result_names:
        archive.write(FINAL_RESULTS / name, arcname=name)
print(f'Attach this file in the Codex task: {RESULT_ARCHIVE}')
